# 3-Way Dual-Convention Evaluation: Phase 8 vs V5_causal vs Noncausal_v4

**Purpose**: Apples-to-apples comparison of all three NZ NIWA-REMS 5 km
precipitation downscaling models on the **same test set** (2012-2013)
with **both evaluation conventions**.

## Two evaluation conventions

| Convention | Threshold strategy | Standard | Used in |
|---|---|---|---|
| **B (pooled)** | Pooled pred+target globally | cGAN literature (CorrDiff) | V5/noncausal existing JSONs |
| **A (ETCCDI)** | Per-pixel climatological p95/p99 from training window | WMO ETCCDI Zhang 2011 | Phase 8 Cell 13 |

Convention B is the one the V5 and noncausal models were originally evaluated
against; Convention A is the WMO-standard used by Phase 8.  Both are computed
here for all three models so neither model is advantaged by convention choice.

## What is new vs the per-model notebooks
- Single shared test batch list, materialised once, iterated 3 times.
- Identical random seed before every sampling loop.
- Both F1 conventions on all models.
- Paired permutation tests + BCa bootstrap CI95 + Holm-Bonferroni correction.
- Unified JSON output with verdict per metric.


In [ ]:
# === Cell 1 : Bootstrap Colab (Drive mount + repo clone/pull + deps) ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'
    ), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'
    ), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'
    ), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric, cftime, h5netcdf, xbatcher, diffusers
    from omegaconf import OmegaConf
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ], check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab - Drive not mounted')

import torch, numpy as np, json, time
import xarray as xr
from omegaconf import OmegaConf

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

_git_sha = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], capture_output=True, text=True
).stdout.strip()
print(f'[bootstrap] git SHA = {_git_sha}  branch = {GIT_BRANCH}')
print(f'[bootstrap] cwd = {os.getcwd()}  torch = {torch.__version__}  cuda = {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[bootstrap] GPU = {torch.cuda.get_device_name(0)}'
          f'  VRAM = {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# === Cell 2 : Constants + pre-registration ===
DRIVE_ROOT = Path('/content/drive/MyDrive/climate_data')
DATA_ROOT  = DRIVE_ROOT / 'data'
HR_RAW_PATH    = DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc'
LR_RAW_PATH    = DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc'
STATIC_PATH    = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'

# Phase 8 checkpoint paths
PHASE8_DIR         = DRIVE_ROOT / 'oracle_9node' / 'phase8_from_scratch' / 'full_19'
PHASE8_STAGE1_CKPT = PHASE8_DIR / 'stage1_last.pth'
PHASE8_STAGE2_CKPT = PHASE8_DIR / 'stage2_last.pth'
PHASE8_CACHE_PATH  = PHASE8_DIR / 'stage1_cache_mu_total.pt'
PHASE8_CLIM_PATH   = PHASE8_DIR / 'clim_train_p95_p99.npz'

# V5_causal checkpoint path
V5_CAUSAL_CKPT  = DRIVE_ROOT / 'ckpt_v2_corrdiff_normal' / 'epoch_last.pth'

# Noncausal_v4 checkpoint path
NONCAUSAL_CKPT  = DRIVE_ROOT / 'ckpt_noncausal' / 'epoch_last.pth'

# Output
OUT_DIR = DRIVE_ROOT / 'oracle_9node' / 'eval_3way_dual_convention'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PHASE8_PRED_CACHE  = OUT_DIR / 'phase8_preds.pt'
V5_PRED_CACHE      = OUT_DIR / 'v5_causal_preds.pt'
NONCAUSAL_PRED_CACHE = OUT_DIR / 'noncausal_v4_preds.pt'
FINAL_JSON         = OUT_DIR / 'results.json'

# Eval hyper-parameters (3-way unified)
SMOKE_MODE    = False    # Set True for a 2-batch sanity run
N_TEST_BATCHES = 2 if SMOKE_MODE else 16
K_SAMPLES      = 4 if SMOKE_MODE else 64
NUM_STEPS      = 8 if SMOKE_MODE else 32
SEED           = 42

# Per-model native CFG scales (from training JSONs)
PHASE8_CFG_SCALE     = 1.0    # Phase 8 Cell 2 CFG_SCALE
V5_CFG_SCALE         = 1.0    # J29 fix: edm_karras does not implement CFG>1.0
NONCAUSAL_CFG_SCALE  = 1.0    # J29 fix: edm_karras does not implement CFG>1.0

# Bootstrap / permutation test
BOOTSTRAP_N_RESAMPLES = 100 if SMOKE_MODE else 1000
PAIRED_PERMUTATION_N  = 500 if SMOKE_MODE else 10000

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hardcoded K9 dates (MEMORY: never capture from CONFIG.data.lr_variables -- mutable)
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

# Hardcoded canonical LR variable list for noncausal (15 vars, Phase 8 memory fix 2026-06-22)
NONCAUSAL_15_VARS = [
    'q_850', 'q_500', 'q_250',
    'u_850', 'u_500', 'u_250',
    'v_850', 'v_500', 'v_250',
    'w_850', 'w_500', 'w_250',
    't_850', 't_500', 't_250',
]

# Phase 8 augmented LR vars (15 canonical + 4 augmented)
AUGMENTED_LR_VARS = NONCAUSAL_15_VARS + [
    'w_700', 'theta_e_850', 'theta_e_500', 'mucape_proxy'
]
AUGMENTED_LR_PATH = PHASE8_DIR / 'lr_augmented_features.nc'

# WMO ETCCDI wet-day threshold
WET_DAY_THRESHOLD_MM = 1.0

# Pre-registration record
PRE_REG = {
    'notebook': '_eval_3way_dual_convention.ipynb',
    'purpose': '3-way apples-to-apples comparison, dual convention',
    'git_sha': _git_sha,
    'git_branch': GIT_BRANCH,
    'seed': SEED,
    'k9_dates': K9_DATES,
    'eval_protocol': {
        'n_test_batches': N_TEST_BATCHES,
        'k_samples': K_SAMPLES,
        'num_steps': NUM_STEPS,
    },
    'checkpoints': {
        'phase8_stage1': str(PHASE8_STAGE1_CKPT),
        'phase8_stage2': str(PHASE8_STAGE2_CKPT),
        'v5_causal': str(V5_CAUSAL_CKPT),
        'noncausal_v4': str(NONCAUSAL_CKPT),
    },
    'cfg_scales': {
        'phase8': PHASE8_CFG_SCALE,
        'v5_causal': V5_CFG_SCALE,
        'noncausal_v4': NONCAUSAL_CFG_SCALE,
    },
    'conventions': {
        'B_pooled': 'compute_f1_extremes without climatology kwarg (cGAN literature)',
        'A_etccdi': 'compute_f1_extremes(climatology=clim_p99) per-pixel (WMO ETCCDI Zhang 2011)',
    },
    'stat_tests': {
        'permutation_n': PAIRED_PERMUTATION_N,
        'bootstrap_n': BOOTSTRAP_N_RESAMPLES,
        'bootstrap_method': 'BCa',
        'correction': 'Holm-Bonferroni',
    },
}

print(f'[Cell 2] DEVICE = {DEVICE}')
print(f'[Cell 2] SMOKE_MODE = {SMOKE_MODE}')
print(f'[Cell 2] N_TEST_BATCHES={N_TEST_BATCHES}  K_SAMPLES={K_SAMPLES}  NUM_STEPS={NUM_STEPS}')
print(f'[Cell 2] phase8 cfg={PHASE8_CFG_SCALE}  v5 cfg={V5_CFG_SCALE}  noncausal cfg={NONCAUSAL_CFG_SCALE}')
print(f'[Cell 2] Pre-registration record captured (SHA {_git_sha[:8]})')


In [ ]:
# === Cell 3 : Config + Phase 8 pipeline + test dataset materialised ONCE ===
# The test batch list is shared by all 3 model sampling loops.
from torch.utils.data import DataLoader as _DataLoader
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES

# Phase 8 CONFIG (base + corrdiff_normal override)
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = False
CONFIG.training.num_workers = 0
OmegaConf.set_struct(CONFIG, False)
CONFIG.data.lr_variables = AUGMENTED_LR_VARS

# Add 9-node metapaths for Phase 8
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in {mm.name for mm in CONFIG.encoder.metapaths}:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))

SEQ_LEN = int(CONFIG.data.seq_len)

pipeline = NetCDFDataPipeline(
    lr_path=str(AUGMENTED_LR_PATH),
    hr_path=str(HR_RAW_PATH),
    static_path=str(STATIC_PATH) if STATIC_PATH.exists() else None,
    seq_len=SEQ_LEN,
    baseline_strategy=str(CONFIG.data.baseline_strategy),
    baseline_factor=int(CONFIG.data.baseline_factor),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG.data.precipitation_delta),
    lr_variables=AUGMENTED_LR_VARS,
    hr_variables=list(CONFIG.data.hr_variables),
    static_variables=list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else [],
    means_path=str(DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc') if (DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc').exists() else None,
    stds_path=str(DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc') if (DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc').exists() else None,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)

test_dataset = pipeline.build_sequence_dataset(
    split='test', seq_len=SEQ_LEN,
    stride=int(CONFIG.data.stride), as_torch=True,
)

# Graph builder (9-node, for Phase 8 + V5_causal)
lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=True,
)
H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])

# IVT / Q / W index helpers (Phase 8 9-node pattern)
_LR_VARS = AUGMENTED_LR_VARS
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]
        u = lr0[:, [_VI[f'u_{lev}']]]
        v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t):
    return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch_phase8(sample, builder, device):
    '''Phase 8 9-node converter (IVT / Q850 / W500 dynamic nodes).'''
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    _ivt = _compute_ivt_nodes(lr0)
    dyn = {}
    for nt in builder.dynamic_node_types:
        if nt == 'Q850':   dyn[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
        elif nt == 'W500': dyn[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
        elif nt == 'IVT':  dyn[nt] = _ensure_2d(_ivt)
        else:              dyn[nt] = _ensure_2d(lr0)
    hetero = builder.prepare_step_data(dyn).to(device)
    return {
        'lr': lr_tensor, 'lr_grid': lr_seq,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero': hetero, 'time': sample.get('time'),
    }

# --- Materialise N_TEST_BATCHES test batches (shared by all 3 models) ---
print(f'[Cell 3] Materialising {N_TEST_BATCHES} test batches...')
_t0 = time.time()
SHARED_BATCHES = []
for _sample in test_dataset:
    if len(SHARED_BATCHES) >= N_TEST_BATCHES:
        break
    SHARED_BATCHES.append(_sample)
print(f'[Cell 3] {len(SHARED_BATCHES)} batches ready in {time.time()-_t0:.1f}s')

# Detect C_LR
_probe = SHARED_BATCHES[0]
C_LR = _probe['lr'].shape[1]
print(f'[Cell 3] C_LR = {C_LR}  H_HR = {H_HR}  W_HR = {W_HR}')
print(f'[Cell 3] Dynamic node types = {builder.dynamic_node_types}')


In [ ]:
# === Cell 4 : Climatology p95/p99 + land mask (Convention A ETCCDI) ===
# Loads pre-computed clim_p95_p99.npz from Phase 8 dir (same training window).
# H5 fix assertion: clim_p99 values must be in [5, 500] mm/day range.

import numpy as np

if PHASE8_CLIM_PATH.exists():
    _clim = np.load(str(PHASE8_CLIM_PATH))
    clim_p99_np  = _clim['clim_p99'].astype(np.float32)
    clim_p95_np  = _clim['clim_p95'].astype(np.float32)
    land_mask_np = _clim['land_mask'].astype(bool)
    print(f'[Cell 4] Loaded climatology from {PHASE8_CLIM_PATH}')
else:
    # Fallback: load from older noncausal eval dir
    _alt_clim = DRIVE_ROOT / 'ckpt_v2_corrdiff_normal' / 'clim_p95_p99.npz'
    if _alt_clim.exists():
        _clim = np.load(str(_alt_clim))
        clim_p99_np  = _clim['clim_p99'].astype(np.float32)
        clim_p95_np  = _clim['clim_p95'].astype(np.float32)
        land_mask_np = _clim['land_mask'].astype(bool)
        print(f'[Cell 4] Loaded climatology (fallback) from {_alt_clim}')
    else:
        raise FileNotFoundError(
            f'Climatology file not found at {PHASE8_CLIM_PATH}\n'
            f'Run Phase 8 Cell 4 first to compute clim_train_p95_p99.npz'
        )

# H5 fix assertion: sanity-check clim_p99 range
_clim_max = float(np.nanmax(clim_p99_np))
_clim_min = float(np.nanmin(clim_p99_np[np.isfinite(clim_p99_np)]))
assert 5 < _clim_max < 500, (
    f'H5 fix FAILED: max(clim_p99) = {_clim_max:.1f} mm/day, expected 5..500. '
    f'Check climatology file units (should be mm/day, not kg/m2/s).'
)
print(f'[Cell 4] clim_p99 range = [{_clim_min:.1f}, {_clim_max:.1f}] mm/day  (H5 check PASSED)')
print(f'[Cell 4] land_mask shape = {land_mask_np.shape}  land_pixels = {land_mask_np.sum()}')

# Torch versions for compute_f1_extremes (Convention A)
clim_p99_torch = torch.from_numpy(clim_p99_np)
clim_p95_torch = torch.from_numpy(clim_p95_np)


In [ ]:
# === Cell 5 : Common metric utilities (shared by all 3 models) ===
import numpy as np
import torch
from scipy.ndimage import uniform_filter
from st_cdgm.evaluation.evaluation_xai import (
    compute_f1_extremes,
    compute_spectrum_distance,
)

PRECIP_DELTA = 0.01  # from pipeline.py precipitation_delta


def to_mm_day(x_log1p: torch.Tensor) -> torch.Tensor:
    '''Convert log1p(pr + delta) -> mm/day, clamped [0, 500].'''
    return (torch.expm1(x_log1p) - PRECIP_DELTA).clamp(min=0.0, max=500.0)


def pearson(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-8) -> float:
    '''Pearson r between two flat 1-D tensors (already finite-filtered).'''
    a_c = a - a.mean()
    b_c = b - b.mean()
    num = (a_c * b_c).sum()
    den = torch.sqrt((a_c * a_c).sum() * (b_c * b_c).sum() + eps)
    return float((num / den).item())


def compute_pearson_global_and_per_sample(pred_full: torch.Tensor,
                                          targets: torch.Tensor):
    '''Pearson global + per-sample average (noncausal_cell_061 lines 244-275 pattern).'''
    _valid = torch.isfinite(targets) & torch.isfinite(pred_full)
    corr_global = float('nan')
    corr_ps_list = []
    try:
        p_flat = pred_full[_valid]
        t_flat = targets[_valid]
        if p_flat.numel() > 1:
            corr_global = pearson(p_flat, t_flat)
        for i in range(pred_full.shape[0]):
            vi = _valid[i]
            if vi.sum() < 2:
                continue
            c = pearson(pred_full[i][vi], targets[i][vi])
            if c == c:
                corr_ps_list.append(c)
    except Exception as _e:
        print(f'  WARNING pearson failed: {_e}')
    corr_ps = float(np.mean(corr_ps_list)) if corr_ps_list else float('nan')
    return corr_global, corr_ps, corr_ps_list


def compute_f1_both_conventions(pred: torch.Tensor,
                                 target: torch.Tensor,
                                 clim_p99: torch.Tensor,
                                 clim_p95: torch.Tensor):
    """
    Returns dict with both Convention B (pooled) and Convention A (ETCCDI per-pixel).
    pred / target expected in mm/day (isfinite, clipped).
    """
    result = {}
    # Convention B: pooled (cGAN standard, what noncausal JSONs used)
    try:
        _b = compute_f1_extremes(pred, target, threshold_percentiles=[95.0, 99.0])
        result['conv_B_F1p99'] = _b.get('p99', float('nan'))
        result['conv_B_F1p95'] = _b.get('p95', float('nan'))
    except Exception as _e:
        print(f'  WARNING conv_B F1 failed: {_e}')
        result['conv_B_F1p99'] = float('nan')
        result['conv_B_F1p95'] = float('nan')
    # Convention A: ETCCDI per-pixel via clim_p99
    try:
        _a99 = compute_f1_extremes(pred, target,
                                    threshold_percentiles=[99.0],
                                    climatology=clim_p99)
        _a95 = compute_f1_extremes(pred, target,
                                    threshold_percentiles=[95.0],
                                    climatology=clim_p95)
        result['conv_A_F1p99'] = _a99.get('p99', float('nan'))
        result['conv_A_F1p95'] = _a95.get('p95', float('nan'))
    except Exception as _e:
        print(f'  WARNING conv_A F1 failed: {_e}')
        result['conv_A_F1p99'] = float('nan')
        result['conv_A_F1p95'] = float('nan')
    return result


def compute_rmse_mae_spread(pred_mean: torch.Tensor,
                             pred_std: torch.Tensor,
                             targets: torch.Tensor):
    _valid = torch.isfinite(targets) & torch.isfinite(pred_mean)
    if not _valid.any():
        return float('nan'), float('nan'), float('nan')
    diff = (pred_mean[_valid] - targets[_valid])
    rmse   = float(diff.pow(2).mean().sqrt().item())
    mae    = float(diff.abs().mean().item())
    spread = float(pred_std[_valid].mean().item()) if pred_std is not None else float('nan')
    return rmse, mae, spread


def compute_rapsd_batch(pred_mean: torch.Tensor,
                         targets: torch.Tensor,
                         n_samples: int = 4) -> float:
    """Average compute_spectrum_distance over up to n_samples batches."""
    dists = []
    for i in range(min(n_samples, pred_mean.shape[0])):
        try:
            d = float(compute_spectrum_distance(pred_mean[i], targets[i]))
            dists.append(d)
        except Exception:
            pass
    return float(np.mean(dists)) if dists else float('nan')


# Convention A: ETCCDI per-pixel F1 (numpy, Phase 8 Cell 13 exact pattern)
def f1_etccdi_per_pixel(pred_mm_np: np.ndarray,
                         target_mm_np: np.ndarray,
                         land_mask: np.ndarray,
                         clim_thresh: np.ndarray) -> dict:
    """Per-pixel ETCCDI F1, land pixels only. Returns {f1, tp, fp, fn}."""
    valid_pix = land_mask & np.isfinite(clim_thresh)
    if not valid_pix.any():
        return {'f1': float('nan'), 'tp': 0, 'fp': 0, 'fn': 0}
    thr = clim_thresh[None, :, :]
    pred_bin   = pred_mm_np >= thr
    target_bin = target_mm_np >= thr
    vb = valid_pix[None, :, :]
    tp = int((pred_bin & target_bin & vb).sum())
    fp = int((pred_bin & ~target_bin & vb).sum())
    fn = int((~pred_bin & target_bin & vb).sum())
    if (tp + fp) == 0 or (tp + fn) == 0:
        return {'f1': 0.0, 'tp': tp, 'fp': fp, 'fn': fn}
    prec = tp / (tp + fp)
    rec  = tp / (tp + fn)
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}


def csi_sedi(pred_bin: np.ndarray, target_bin: np.ndarray) -> tuple:
    """CSI and SEDI (Phase 8 Cell 13 pattern)."""
    tp = int((pred_bin & target_bin).sum())
    fp = int((pred_bin & ~target_bin).sum())
    fn = int((~pred_bin & target_bin).sum())
    tn = int((~pred_bin & ~target_bin).sum())
    csi = tp / max(tp + fp + fn, 1)
    h = tp / max(tp + fn, 1)
    f = fp / max(fp + tn, 1)
    eps = 1e-12
    h = max(min(h, 1 - eps), eps)
    f = max(min(f, 1 - eps), eps)
    num = np.log(f) - np.log(h) - np.log(1 - f) + np.log(1 - h)
    den = np.log(f) + np.log(h) + np.log(1 - f) + np.log(1 - h)
    sedi = float(num / den) if den != 0 else float('nan')
    return csi, sedi


def fss_neighborhood(pred_bin: np.ndarray,
                      target_bin: np.ndarray,
                      n: int = 9, mode: str = 'reflect') -> float:
    """FSS at neighborhood scale n (Phase 8 Cell 13 pattern)."""
    P = pred_bin.astype(np.float32)
    Q = target_bin.astype(np.float32)
    Pf = np.stack([uniform_filter(P[i], size=n, mode=mode) for i in range(P.shape[0])])
    Qf = np.stack([uniform_filter(Q[i], size=n, mode=mode) for i in range(Q.shape[0])])
    mse  = float(((Pf - Qf) ** 2).mean())
    norm = float((Pf ** 2 + Qf ** 2).mean())
    return 1.0 - mse / max(norm, 1e-12)


def drizzle_bias(pred_mm_np: np.ndarray,
                  target_mm_np: np.ndarray,
                  drizzle_thr: float = 1.0) -> float:
    """Mean drizzle bias: M5 fix with isfinite filter."""
    valid = np.isfinite(pred_mm_np) & np.isfinite(target_mm_np)
    if not valid.any():
        return float('nan')
    pred_drizzle   = ((pred_mm_np[valid]   > 0) & (pred_mm_np[valid]   < drizzle_thr)).mean()
    target_drizzle = ((target_mm_np[valid] > 0) & (target_mm_np[valid] < drizzle_thr)).mean()
    return float(pred_drizzle - target_drizzle)


def compute_climate_indices_bias(pred_mm_np: np.ndarray,
                                  target_mm_np: np.ndarray,
                                  land_mask: np.ndarray,
                                  n_batches: int) -> dict:
    """Rx1day_bias, R10_bias, CDD_bias (Phase 8 Cell 13).  N-aware label."""
    rx1d_pred   = float(np.nanmean(pred_mm_np.max(axis=0)[land_mask]))
    rx1d_target = float(np.nanmean(target_mm_np.max(axis=0)[land_mask]))
    r10_pred    = float(np.nanmean((pred_mm_np >= 10).sum(axis=0)[land_mask]))
    r10_target  = float(np.nanmean((target_mm_np >= 10).sum(axis=0)[land_mask]))
    cdd_pred    = float(np.nanmean((pred_mm_np < 1).sum(axis=0)[land_mask]))
    cdd_target  = float(np.nanmean((target_mm_np < 1).sum(axis=0)[land_mask]))
    return {
        f'Rx1day_bias_N{n_batches}': rx1d_pred - rx1d_target,
        f'R10_bias_N{n_batches}':    r10_pred  - r10_target,
        f'CDD_bias_N{n_batches}':    cdd_pred  - cdd_target,
    }


def _persist_load_state_dict(model, state_dict, strict: bool = False):
    """Load state_dict with strict=False, log missing/unexpected keys."""
    if state_dict is None:
        print(f'  WARNING: state_dict is None for {model.__class__.__name__}')
        return
    out = model.load_state_dict(state_dict, strict=strict)
    if out.missing_keys:
        print(f'  {model.__class__.__name__} missing keys: {len(out.missing_keys)}')
    if out.unexpected_keys:
        print(f'  {model.__class__.__name__} unexpected keys: {len(out.unexpected_keys)}')


print('[Cell 5] Common metric utilities defined')
print('  Functions: pearson, compute_f1_both_conventions, compute_rmse_mae_spread,')
print('  compute_rapsd_batch, f1_etccdi_per_pixel, csi_sedi, fss_neighborhood,')
print('  drizzle_bias, compute_climate_indices_bias, _persist_load_state_dict')


In [ ]:
# === Cell 6 : Phase 8 BUILD (Stage 1 + Stage 2 with best EMA) ===
import copy
from torch.nn.parameter import UninitializedParameter as _UP
from st_cdgm.models.dual_path_stage1 import DualPathPredictor
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from st_cdgm.training.stage1_paths import batch_lr_grid_last

# ---- Stage 1 architecture ----
_p8_metapath_configs = [
    IntelligibleVariableConfig(
        name=m.name, meta_path=(m.src, m.relation, m.target), pool='mean'
    )
    for m in CONFIG.encoder.metapaths
]
p8_encoder = IntelligibleVariableEncoder(
    configs=_p8_metapath_configs,
    hidden_dim=int(CONFIG.encoder.hidden_dim),
    conditioning_dim=int(CONFIG.encoder.conditioning_dim),
).to(DEVICE)
p8_num_vars = len(_p8_metapath_configs)

_p8_probe_lr = builder.lr_grid_to_nodes(SHARED_BATCHES[0]['lr'][0])
p8_rcn_driver_dim = _p8_probe_lr.shape[-1]
p8_rcn_cell = RCNCell(
    num_vars=p8_num_vars, hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=p8_rcn_driver_dim, reconstruction_dim=p8_rcn_driver_dim,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
p8_rcn_runner = RCNSequenceRunner(
    p8_rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval')
)

_rh_cfg = CONFIG.two_stage.regression_head
p8_regression_head = GraphToGridDecoder(
    d_model=int(_rh_cfg.d_model), hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(_rh_cfg.intermediate_h),
    intermediate_w=int(_rh_cfg.intermediate_w),
    n_heads=int(_rh_cfg.n_heads),
    refine_channels=int(_rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

p8_dual_path = DualPathPredictor(
    in_channels=C_LR, base_ch=48, hr_h=H_HR, hr_w=W_HR,
    gate_max_mean=0.40, path_b_kind='unet',
    path_b_unet_channels=(32, 64, 128),
    path_b_unet_lr_shape=(23, 26),
).to(DEVICE)

p8_alpha_logit = torch.nn.Parameter(torch.tensor(0.0, device=DEVICE))

# Load Stage 1 checkpoint
print(f'[Cell 6] Loading Stage 1 from {PHASE8_STAGE1_CKPT}')
_ck_s1 = torch.load(str(PHASE8_STAGE1_CKPT), map_location=DEVICE, weights_only=False)
_persist_load_state_dict(p8_encoder, _ck_s1.get('encoder_state_dict'))
_persist_load_state_dict(p8_rcn_cell, _ck_s1.get('rcn_cell_state_dict'))
_persist_load_state_dict(p8_regression_head, _ck_s1.get('regression_head_state_dict'))
_persist_load_state_dict(p8_dual_path, _ck_s1.get('dual_path_state_dict'))
if _ck_s1.get('alpha_logit') is not None:
    p8_alpha_logit.data = _ck_s1['alpha_logit'].to(DEVICE)
print(f'[Cell 6] Stage 1 loaded (epoch {_ck_s1.get("epoch")})')

# Freeze Stage 1 — guard against LazyModule UninitializedParameter in fallback metapaths
for m in [p8_encoder, p8_rcn_cell, p8_regression_head, p8_dual_path]:
    for p in m.parameters():
        if isinstance(p, _UP):
            continue
        p.requires_grad_(False)
    m.eval()

# ---- Stage 2 architecture ----
_UNET_KW = OmegaConf.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in _UNET_KW and isinstance(_UNET_KW[_k], list):
        _UNET_KW[_k] = tuple(_UNET_KW[_k])
_UNET_KW['projection_class_embeddings_input_dim'] = (
    p8_num_vars * int(CONFIG.diffusion.conditioning_dim)
)
_p8_edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))
SIGMA_DATA_P8 = 0.193  # Phase 8 Cell 2 recalibrated value

_p8_probe_hr = SHARED_BATCHES[0]['residual']
p8_hr_channels = int(_p8_probe_hr.shape[1])

def _build_p8_decoder():
    d = CausalDiffusionDecoder(
        in_channels=p8_hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=int(CONFIG.diffusion.height), width=int(CONFIG.diffusion.width),
        unet_kwargs=_UNET_KW,
        scheduler_type=str(CONFIG.diffusion.scheduler_type),
        use_gradient_checkpointing=False,
        conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
        anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
        edm_config=_p8_edm_cfg, causal_concat=True,
    ).to(DEVICE)
    d.edm_config.sigma_data = SIGMA_DATA_P8
    return d

EMA_DECAYS_P8 = [0.999, 0.9995, 0.9999]

# Load Stage 2 checkpoint + multi-EMA
print(f'[Cell 6] Loading Stage 2 from {PHASE8_STAGE2_CKPT}')
_ck_s2 = torch.load(str(PHASE8_STAGE2_CKPT), map_location=DEVICE, weights_only=False)
p8_live_decoder = _build_p8_decoder()
_persist_load_state_dict(p8_live_decoder, _ck_s2.get('diffusion_state_dict'))

p8_ema_decoders = []
_saved_ema_sds = _ck_s2.get('ema_state_dicts', [])
for i, decay in enumerate(EMA_DECAYS_P8):
    ema = _build_p8_decoder()
    if i < len(_saved_ema_sds):
        _persist_load_state_dict(ema, _saved_ema_sds[i])
    else:
        ema.load_state_dict(p8_live_decoder.state_dict())
    ema.eval()
    for param in ema.parameters():
        param.requires_grad_(False)
    ema._ema_decay = decay
    p8_ema_decoders.append(ema)
if _ck_s2.get('alpha_logit') is not None:
    p8_alpha_logit.data = _ck_s2['alpha_logit'].to(DEVICE)

print(f'[Cell 6] Stage 2 loaded (epoch {_ck_s2.get("epoch")})  '
      f'alpha_logit -> alpha = {float(torch.sigmoid(p8_alpha_logit)):.4f}')

# Post-hoc EMA sweep on a small subset to pick best EMA
def _p8_ema_select(ema_list, samples_sub, n_check=2):
    """Quick-sweep EMA decoders, return index with lowest residual RMSE."""
    scores = []
    for ema in ema_list:
        rmses = []
        with torch.no_grad():
            for sample in samples_sub[:n_check]:
                batch = convert_sample_to_batch_phase8(sample, builder, DEVICE)
                lr_d  = batch['lr'].to(DEVICE)
                h_in  = p8_encoder.init_state(batch['hetero']).to(DEVICE)
                drivers = [lr_d[t] for t in range(lr_d.shape[0])]
                seq_out = p8_rcn_runner.run(h_in, drivers, reconstruction_sources=None)
                mu_A = p8_regression_head(seq_out.states[-1])
                if mu_A.dim() == 3:
                    mu_A = mu_A.unsqueeze(0)
                lr_grid = batch_lr_grid_last(batch, builder=builder, device=DEVICE)
                lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
                mu_total, _, _ = p8_dual_path(lr_safe, mu_A)
                mu_total = torch.nan_to_num(mu_total, nan=0.0)
                bl = batch['baseline'][-1].to(DEVICE)
                if bl.dim() == mu_total.dim() - 1:
                    bl = bl.unsqueeze(0)
                bl = torch.nan_to_num(bl, nan=0.0)
                tgt = batch['residual'][-1].to(DEVICE)
                if tgt.dim() == 3:
                    tgt = tgt.unsqueeze(0)
                out = ema.sample(
                    conditioning=None,
                    num_steps=min(8, NUM_STEPS),
                    scheduler_type='edm_karras',
                    cfg_scale=0.0,
                    apply_constraints=False,
                    mu_HR=mu_total, baseline_log=bl,
                )
                res = out.residual if hasattr(out, 'residual') else out
                valid = torch.isfinite(tgt)
                _rmse = float(((res[valid] - tgt[valid]) ** 2).mean().sqrt())
                rmses.append(_rmse)
        scores.append(float(np.mean(rmses)) if rmses else float('inf'))
    best_i = int(np.argmin(scores))
    print(f'[Cell 6] EMA sweep RMSEs: {[f"{s:.5f}" for s in scores]}  '
          f'-> best decay = {EMA_DECAYS_P8[best_i]}')
    return p8_ema_decoders[best_i], EMA_DECAYS_P8[best_i]

_n_ema_check = 1 if SMOKE_MODE else 3
p8_best_ema, p8_best_decay = _p8_ema_select(p8_ema_decoders, SHARED_BATCHES, _n_ema_check)
print(f'[Cell 6] Phase 8 build complete. Using EMA decay = {p8_best_decay}')


In [ ]:
# === Cell 7 : Phase 8 SAMPLE (idempotent) ===
from st_cdgm.training.stage1_paths import batch_lr_grid_last

if PHASE8_PRED_CACHE.exists() and not SMOKE_MODE:
    print(f'[Cell 7] Loading cached Phase 8 predictions from {PHASE8_PRED_CACHE}')
    _p8_cache = torch.load(str(PHASE8_PRED_CACHE), map_location='cpu', weights_only=False)
    p8_pred_mm      = _p8_cache['pred_mm']
    p8_target_mm    = _p8_cache['target_mm']
    p8_pred_log     = _p8_cache['pred_full_log']
    p8_target_log   = _p8_cache['target_full_log']
    p8_pred_std     = _p8_cache['pred_std']
    p8_per_batch_rmse = _p8_cache['per_batch_rmse']
    print(f'[Cell 7] Loaded {p8_pred_mm.shape[0]} Phase 8 batches from cache')
else:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    _p8_means, _p8_stds, _p8_tgts, _p8_mus, _p8_baselines = [], [], [], [], []
    _p8_per_batch_rmse = []
    _t0 = time.time()

    with torch.no_grad():
        for _bi, _sample in enumerate(SHARED_BATCHES):
            batch = convert_sample_to_batch_phase8(_sample, builder, DEVICE)

            # Stage 1 forward
            lr_d = batch['lr'].to(DEVICE)
            h_in = p8_encoder.init_state(batch['hetero']).to(DEVICE)
            drivers = [lr_d[t] for t in range(lr_d.shape[0])]
            seq_out = p8_rcn_runner.run(h_in, drivers, reconstruction_sources=None)
            mu_A = p8_regression_head(seq_out.states[-1])
            if mu_A.dim() == 3:
                mu_A = mu_A.unsqueeze(0)
            lr_grid = batch_lr_grid_last(batch, builder=builder, device=DEVICE)
            lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
            mu_total, _, _ = p8_dual_path(lr_safe, mu_A)
            mu_total = torch.nan_to_num(mu_total, nan=0.0)
            bl = batch['baseline'][-1].to(DEVICE)
            if bl.dim() == mu_total.dim() - 1:
                bl = bl.unsqueeze(0)
            bl = torch.nan_to_num(bl, nan=0.0)
            tgt = batch['residual'][-1].to(DEVICE)
            if tgt.dim() == 3:
                tgt = tgt.unsqueeze(0)

            # Stage 2 sampling (K_SAMPLES realisations)
            _samples_k = []
            for _k in range(K_SAMPLES):
                torch.manual_seed(SEED + 1000 * _bi + _k)
                out = p8_best_ema.sample(
                    conditioning=None,
                    num_steps=NUM_STEPS,
                    scheduler_type='edm_karras',
                    cfg_scale=PHASE8_CFG_SCALE,
                    apply_constraints=False,
                    mu_HR=mu_total, baseline_log=bl,
                )
                res = out.residual if hasattr(out, 'residual') else out
                _samples_k.append(res)

            _stack = torch.stack(_samples_k, dim=0)
            _mean  = _stack.mean(dim=0)
            _std   = _stack.std(dim=0)

            _p8_means.append(_mean.cpu())
            _p8_stds.append(_std.cpu())
            _p8_tgts.append(tgt.cpu())
            _p8_mus.append(mu_total.cpu())
            _p8_baselines.append(bl.cpu())

            # Per-batch RMSE on residual (for stat tests)
            _v = torch.isfinite(tgt)
            _rmse_b = float(((_mean[_v] - tgt[_v]) ** 2).mean().sqrt())
            _p8_per_batch_rmse.append(_rmse_b)

            if (_bi + 1) % 4 == 0 or (_bi + 1) == len(SHARED_BATCHES):
                print(f'  [P8] batch {_bi+1}/{len(SHARED_BATCHES)} '
                      f'elapsed={time.time()-_t0:.0f}s  '
                      f'rmse_residual={_rmse_b:.5f}')

    p8_pred_residual  = torch.cat(_p8_means, dim=0)
    p8_pred_std       = torch.cat(_p8_stds, dim=0)
    p8_target_residual = torch.cat(_p8_tgts, dim=0)
    p8_mu_total       = torch.cat(_p8_mus, dim=0)
    p8_baseline_cpu   = torch.cat(_p8_baselines, dim=0)

    # Full log1p reconstruction (Phase 8 Cell 12 pattern)
    _alpha_final = float(torch.sigmoid(p8_alpha_logit).detach())
    p8_pred_log   = p8_baseline_cpu + _alpha_final * p8_mu_total + p8_pred_residual
    p8_target_log = p8_baseline_cpu + p8_mu_total  + p8_target_residual

    # Convert to mm/day
    p8_pred_mm   = to_mm_day(p8_pred_log)
    p8_target_mm = to_mm_day(p8_target_log)
    p8_per_batch_rmse = _p8_per_batch_rmse

    if not SMOKE_MODE:
        torch.save({
            'pred_mm': p8_pred_mm, 'target_mm': p8_target_mm,
            'pred_full_log': p8_pred_log, 'target_full_log': p8_target_log,
            'pred_std': p8_pred_std,
            'per_batch_rmse': p8_per_batch_rmse,
            'alpha_final': _alpha_final,
        }, str(PHASE8_PRED_CACHE))

    print(f'[Cell 7] Phase 8 sampling done in {time.time()-_t0:.0f}s')
    print(f'[Cell 7] alpha_final = {_alpha_final:.4f}')
    print(f'[Cell 7] pred_mm shape = {tuple(p8_pred_mm.shape)}')
    print(f'[Cell 7] pred_mm range: [{float(p8_pred_mm.min()):.2f}, {float(p8_pred_mm.max()):.2f}]')


In [ ]:
# === Cell 8 : V5_causal BUILD (IntelligibleVariableEncoder + RCN + diffusion, causal_concat=True) ===
# NOTE: V5_causal was trained on the ORIGINAL 15 LR vars (not augmented 19).
# We build a separate pipeline with 15 vars and a standard (non-9-node) builder.
from torch.utils.data import DataLoader as _DataLoader
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig

# V5 uses base config + corrdiff_normal (no 9-node augmentation)
CONFIG_V5 = OmegaConf.load('config/training_config.yaml')
_corrdiff_v5 = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG_V5 = OmegaConf.merge(CONFIG_V5, _corrdiff_v5)
OmegaConf.set_struct(CONFIG_V5, False)
CONFIG_V5.data.lr_variables = NONCAUSAL_15_VARS  # hardcoded canonical list

SEQ_LEN_V5 = int(CONFIG_V5.data.seq_len)
pipeline_v5 = NetCDFDataPipeline(
    lr_path=str(LR_RAW_PATH),
    hr_path=str(HR_RAW_PATH),
    static_path=str(STATIC_PATH) if STATIC_PATH.exists() else None,
    seq_len=SEQ_LEN_V5,
    baseline_strategy=str(CONFIG_V5.data.baseline_strategy),
    baseline_factor=int(CONFIG_V5.data.baseline_factor),
    normalize=bool(CONFIG_V5.data.normalize),
    nan_fill_strategy=str(CONFIG_V5.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG_V5.data.precipitation_delta),
    lr_variables=NONCAUSAL_15_VARS,
    hr_variables=list(CONFIG_V5.data.hr_variables),
    static_variables=list(CONFIG_V5.data.static_variables) if CONFIG_V5.data.get('static_variables') else [],
    means_path=str(DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc') if (DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc').exists() else None,
    stds_path=str(DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc') if (DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc').exists() else None,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)

# Standard (non-9-node) builder for V5
builder_v5 = HeteroGraphBuilder(
    lr_shape=tuple(CONFIG_V5.graph.lr_shape),
    hr_shape=tuple(CONFIG_V5.graph.hr_shape),
    static_dataset=pipeline_v5.get_static_dataset(),
    include_mid_layer=CONFIG_V5.graph.include_mid_layer,
    extended_9node=False,  # V5 was trained without 9-node extension
)

test_dataset_v5 = pipeline_v5.build_sequence_dataset(
    split='test', seq_len=SEQ_LEN_V5,
    stride=int(CONFIG_V5.data.stride), as_torch=True,
)

# Materialise matching test batches for V5 (same N, same temporal order)
print(f'[Cell 8] Materialising {N_TEST_BATCHES} V5 test batches...')
V5_BATCHES = []
for _s in test_dataset_v5:
    if len(V5_BATCHES) >= N_TEST_BATCHES:
        break
    V5_BATCHES.append(_s)
print(f'[Cell 8] V5 batches: {len(V5_BATCHES)}')

# Build encoder from CONFIG_V5 metapaths
_v5_allowed = set(builder_v5.dynamic_node_types) | set(builder_v5.static_node_types)
v5_encoder_configs = []
for _mp in CONFIG_V5.encoder.metapaths:
    if _mp.src in _v5_allowed and _mp.target in _v5_allowed:
        v5_encoder_configs.append(IntelligibleVariableConfig(
            name=_mp.name, meta_path=(_mp.src, _mp.relation, _mp.target),
            pool=_mp.get('pool', 'mean'),
        ))
if pipeline_v5.get_static_dataset() is not None:
    v5_encoder_configs.append(IntelligibleVariableConfig(
        name='static', meta_path=('SP_HR', 'causes', 'GP850'), pool='mean',
    ))

v5_encoder = IntelligibleVariableEncoder(
    configs=v5_encoder_configs,
    hidden_dim=int(CONFIG_V5.encoder.hidden_dim),
    conditioning_dim=int(CONFIG_V5.encoder.conditioning_dim),
).to(DEVICE)
v5_num_vars = len(v5_encoder_configs)

_v5_probe_lr = builder_v5.lr_grid_to_nodes(V5_BATCHES[0]['lr'][0])
v5_rcn_driver_dim = _v5_probe_lr.shape[-1]
v5_rcn_cell = RCNCell(
    num_vars=v5_num_vars, hidden_dim=int(CONFIG_V5.rcn.hidden_dim),
    driver_dim=v5_rcn_driver_dim, reconstruction_dim=v5_rcn_driver_dim,
    dropout=float(CONFIG_V5.rcn.dropout),
).to(DEVICE)
v5_rcn_runner = RCNSequenceRunner(
    v5_rcn_cell, detach_interval=CONFIG_V5.rcn.get('detach_interval')
)

_v5_rh_cfg = CONFIG_V5.two_stage.regression_head
v5_regression_head = GraphToGridDecoder(
    d_model=int(_v5_rh_cfg.d_model),
    hr_h=int(CONFIG_V5.diffusion.height), hr_w=int(CONFIG_V5.diffusion.width),
    intermediate_h=int(_v5_rh_cfg.intermediate_h),
    intermediate_w=int(_v5_rh_cfg.intermediate_w),
    n_heads=int(_v5_rh_cfg.n_heads),
    refine_channels=int(_v5_rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

# Diffusion decoder for V5
_v5_unet_kw = OmegaConf.to_container(CONFIG_V5.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in _v5_unet_kw and isinstance(_v5_unet_kw[_k], list):
        _v5_unet_kw[_k] = tuple(_v5_unet_kw[_k])
_v5_unet_kw['projection_class_embeddings_input_dim'] = (
    v5_num_vars * int(CONFIG_V5.diffusion.conditioning_dim)
)
_v5_edm_cfg = EDMConfig.from_yaml_dict(CONFIG_V5.diffusion.get('edm', {}))
_v5_hr_channels = int(V5_BATCHES[0]['residual'].shape[1])

v5_diffusion = CausalDiffusionDecoder(
    in_channels=_v5_hr_channels,
    conditioning_dim=int(CONFIG_V5.diffusion.conditioning_dim),
    height=int(CONFIG_V5.diffusion.height), width=int(CONFIG_V5.diffusion.width),
    unet_kwargs=_v5_unet_kw,
    scheduler_type=str(CONFIG_V5.diffusion.scheduler_type),
    use_gradient_checkpointing=False,
    conv_padding_mode=str(CONFIG_V5.diffusion.get('conv_padding_mode', 'zeros')),
    anti_checkerboard=bool(CONFIG_V5.diffusion.get('anti_checkerboard', False)),
    edm_config=_v5_edm_cfg, causal_concat=True,
).to(DEVICE)

# Load V5 checkpoint (BS37 EMA pattern)
print(f'[Cell 8] Loading V5_causal from {V5_CAUSAL_CKPT}')
_ck_v5 = torch.load(str(V5_CAUSAL_CKPT), map_location=DEVICE, weights_only=False)
print(f'  epoch={_ck_v5.get("epoch")} val_loss={_ck_v5.get("val_loss")}')
_persist_load_state_dict(v5_encoder, _ck_v5.get('encoder_state_dict'))
_persist_load_state_dict(v5_rcn_cell, _ck_v5.get('rcn_cell_state_dict'))
_persist_load_state_dict(v5_regression_head, _ck_v5.get('regression_head_state_dict'))
# BS37 EMA: prefer EMA weights for inference
_v5_ema_sd = _ck_v5.get('diffusion_ema_state_dict') or _ck_v5.get('ema_state_dict')
if _v5_ema_sd is not None:
    print('  BS37 EMA detected -- loading EMA weights for V5 inference')
    _persist_load_state_dict(v5_diffusion, _v5_ema_sd)
else:
    print('  No EMA state_dict found -- loading live diffusion weights for V5')
    _persist_load_state_dict(v5_diffusion, _ck_v5.get('diffusion_state_dict'))

# Detect causal_concat on loaded model
_v5_core = getattr(v5_diffusion, '_orig_mod', v5_diffusion)
_v5_core = getattr(_v5_core, 'module', _v5_core)
v5_causal_concat = bool(getattr(_v5_core, 'causal_concat', True))
print(f'  V5 causal_concat = {v5_causal_concat}')

for m in [v5_encoder, v5_rcn_cell, v5_regression_head, v5_diffusion]:
    m.eval()
    for _p in m.parameters():
        _p.requires_grad_(False)

print(f'[Cell 8] V5_causal build complete')


In [ ]:
# === Cell 9 : V5_causal SAMPLE (idempotent) ===
from st_cdgm.evaluation.two_stage_inference import build_two_stage_inputs
from st_cdgm.training.stage1_paths import resolve_run_variant

_v5_run_variant = 'causal'  # V5_causal always causal

def _v5_build_inputs(batch):
    return build_two_stage_inputs(
        batch,
        variant=_v5_run_variant,
        regression_head=v5_regression_head,
        encoder=v5_encoder,
        rcn_runner=v5_rcn_runner,
        builder=builder_v5,
        device=DEVICE,
    )

def _v5_sample_once(mu_HR, baseline_log):
    kw = dict(
        num_steps=NUM_STEPS,
        scheduler_type='edm_karras',
        cfg_scale=V5_CFG_SCALE,
        apply_constraints=False,
    )
    if v5_causal_concat:
        kw['mu_HR'] = mu_HR
        kw['baseline_log'] = baseline_log
    return _v5_core.sample(conditioning=None, **kw).residual

if V5_PRED_CACHE.exists() and not SMOKE_MODE:
    print(f'[Cell 9] Loading cached V5_causal predictions from {V5_PRED_CACHE}')
    _v5_cache = torch.load(str(V5_PRED_CACHE), map_location='cpu', weights_only=False)
    v5_pred_mm      = _v5_cache['pred_mm']
    v5_target_mm    = _v5_cache['target_mm']
    v5_pred_log     = _v5_cache['pred_full_log']
    v5_target_log   = _v5_cache['target_full_log']
    v5_pred_std     = _v5_cache['pred_std']
    v5_per_batch_rmse = _v5_cache['per_batch_rmse']
    print(f'[Cell 9] Loaded {v5_pred_mm.shape[0]} V5_causal batches from cache')
else:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    _v5_means, _v5_stds, _v5_tgts, _v5_mus, _v5_bls, _v5_per_rmse = [], [], [], [], [], []
    _t0 = time.time()

    with torch.no_grad():
        for _bi, _sample in enumerate(V5_BATCHES):
            # Build batch for V5 (using V5 builder)
            from st_cdgm.evaluation.evaluation_xai import convert_sample_to_batch as _csb_v5
            _batch = _csb_v5(_sample, builder_v5, DEVICE)
            _cond, _mu_HR, _bl, _tgt = _v5_build_inputs(_batch)

            _samples_k = []
            for _k in range(K_SAMPLES):
                torch.manual_seed(SEED + 2000 * _bi + _k)
                _res = _v5_sample_once(_mu_HR, _bl)
                _samples_k.append(_res)

            _stack = torch.stack(_samples_k, dim=0)
            _mean  = _stack.mean(dim=0)
            _std   = _stack.std(dim=0)

            _v5_means.append(_mean.cpu())
            _v5_stds.append(_std.cpu())
            _v5_tgts.append(_tgt.cpu())
            _v5_mus.append(_mu_HR.cpu() if _mu_HR is not None else torch.zeros_like(_mean.cpu()))
            _v5_bls.append(_bl.cpu())

            _v5_per_rmse.append(
                float(((_mean - _tgt).pow(2)[torch.isfinite(_tgt)].mean().sqrt()).item())
            )

            if (_bi + 1) % 4 == 0 or (_bi + 1) == len(V5_BATCHES):
                print(f'  [V5] batch {_bi+1}/{len(V5_BATCHES)} '
                      f'elapsed={time.time()-_t0:.0f}s')

    v5_pred_res  = torch.cat(_v5_means, dim=0)
    v5_pred_std  = torch.cat(_v5_stds, dim=0)
    v5_tgt_res   = torch.cat(_v5_tgts, dim=0)
    v5_mu_cat    = torch.cat(_v5_mus, dim=0)
    v5_bl_cat    = torch.cat(_v5_bls, dim=0)

    # BS31f: full prediction = mu_HR + delta (causal_concat mode)
    if v5_causal_concat:
        _v5_valid_mu = torch.isfinite(v5_mu_cat)
        if _v5_valid_mu.all():
            v5_pred_full = v5_mu_cat + v5_pred_res
        else:
            v5_pred_full = v5_pred_res
    else:
        v5_pred_full = v5_pred_res

    v5_pred_log  = v5_bl_cat + v5_pred_full
    v5_target_log = v5_bl_cat + v5_tgt_res + v5_mu_cat  # note: tgt_res = HR_log - bl - mu_HR

    v5_pred_mm   = to_mm_day(v5_pred_log)
    v5_target_mm = to_mm_day(v5_target_log)
    v5_per_batch_rmse = _v5_per_rmse

    if not SMOKE_MODE:
        torch.save({
            'pred_mm': v5_pred_mm, 'target_mm': v5_target_mm,
            'pred_full_log': v5_pred_log, 'target_full_log': v5_target_log,
            'pred_std': v5_pred_std, 'per_batch_rmse': v5_per_batch_rmse,
        }, str(V5_PRED_CACHE))

    print(f'[Cell 9] V5_causal sampling done in {time.time()-_t0:.0f}s')
    print(f'[Cell 9] pred_mm shape = {tuple(v5_pred_mm.shape)}')


In [ ]:
# === Cell 10 : noncausal_v4 BUILD (architecture inferred from checkpoint weights) ===
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from st_cdgm.models.regression_mean_predictor import RegressionMeanPredictor, RegressionPredictorConfig
from st_cdgm.models.regression_head import GraphToGridDecoder
from st_cdgm.models.intelligible_encoder import IntelligibleVariableEncoder, IntelligibleVariableConfig
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.training.stage1_paths import resolve_run_variant


def _sd_tensor(sd, suffix):
    """Find a tensor in a state_dict by suffix (tolerates compile/DDP prefixes)."""
    if sd is None:
        return None
    for k, v in sd.items():
        if k.endswith(suffix):
            return v
    return None


def _infer_unet_arch_from_sd(sd):
    """Return (causal_concat, projection_class_embeddings_input_dim) from UNet weights."""
    conv_w = _sd_tensor(sd, 'unet.conv_in.weight')
    proj_w = _sd_tensor(sd, 'unet.class_embedding.linear_1.weight')
    causal_concat = bool(conv_w is not None and int(conv_w.shape[1]) == 3)
    proj_dim = int(proj_w.shape[1]) if proj_w is not None else None
    return causal_concat, proj_dim


def _is_regression_mean_predictor_sd(sd):
    if not sd:
        return False
    keys = ' '.join(sd.keys())
    return ('hr_proj' in keys) or ('unet.down_blocks' in keys and 'cross_attn' not in keys)


# Noncausal config (15-var canonical LR list, two_stage.run_variant=noncausal)
CONFIG_NC = OmegaConf.load('config/training_config.yaml')
CONFIG_NC = OmegaConf.merge(CONFIG_NC, OmegaConf.load('config/training_config_noncausal.yaml'))
OmegaConf.set_struct(CONFIG_NC, False)
CONFIG_NC.data.lr_variables = NONCAUSAL_15_VARS

print(f'[Cell 10] Materialising {N_TEST_BATCHES} noncausal_v4 test batches...')
NC_BATCHES = []
for _s in test_dataset_v5:
    if len(NC_BATCHES) >= N_TEST_BATCHES:
        break
    NC_BATCHES.append(_s)
print(f'[Cell 10] noncausal_v4 batches: {len(NC_BATCHES)}')

nc_hr_channels = int(NC_BATCHES[0]['residual'].shape[1])

print(f'[Cell 10] Loading noncausal_v4 ckpt: {NONCAUSAL_CKPT}')
_ck_nc = torch.load(str(NONCAUSAL_CKPT), map_location=DEVICE, weights_only=False)
print(f'  keys: {[k for k in _ck_nc.keys() if "state" in k or "epoch" in k]}')
print(f'  epoch={_ck_nc.get("epoch")} val_loss={_ck_nc.get("val_loss")}')

_ck_cfg = (_ck_nc.get('config_full') or _ck_nc.get('config') or {})
_nc_run_variant = str(
    (_ck_cfg.get('two_stage', {}) or {}).get('run_variant')
    or resolve_run_variant(CONFIG_NC)
).lower()
print(f'  run_variant (checkpoint/config) = {_nc_run_variant}')

_nc_diff_sd = (
    _ck_nc.get('diffusion_ema_state_dict')
    or _ck_nc.get('ema_state_dict')
    or _ck_nc.get('diffusion_state_dict')
)
nc_causal_concat, _nc_proj_dim = _infer_unet_arch_from_sd(_nc_diff_sd)
print(f'  causal_concat (from conv_in) = {nc_causal_concat}')
print(f'  projection_class_embeddings_input_dim (from ckpt) = {_nc_proj_dim}')

_nc_unet_kw = OmegaConf.to_container(CONFIG_NC.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in _nc_unet_kw and isinstance(_nc_unet_kw[_k], list):
        _nc_unet_kw[_k] = tuple(_nc_unet_kw[_k])
if _nc_proj_dim is not None:
    _nc_unet_kw['projection_class_embeddings_input_dim'] = _nc_proj_dim

_nc_edm_cfg = EDMConfig.from_yaml_dict(CONFIG_NC.diffusion.get('edm', {}))
nc_diffusion = CausalDiffusionDecoder(
    in_channels=nc_hr_channels,
    conditioning_dim=int(CONFIG_NC.diffusion.conditioning_dim),
    height=int(CONFIG_NC.diffusion.height), width=int(CONFIG_NC.diffusion.width),
    unet_kwargs=_nc_unet_kw,
    scheduler_type=str(CONFIG_NC.diffusion.scheduler_type),
    use_gradient_checkpointing=False,
    conv_padding_mode=str(CONFIG_NC.diffusion.get('conv_padding_mode', 'zeros')),
    anti_checkerboard=bool(CONFIG_NC.diffusion.get('anti_checkerboard', False)),
    edm_config=_nc_edm_cfg,
    causal_concat=nc_causal_concat,
).to(DEVICE)

_nc_ema_sd = _ck_nc.get('diffusion_ema_state_dict') or _ck_nc.get('ema_state_dict')
if _nc_ema_sd is not None:
    print('  BS37 EMA detected -- loading EMA weights for noncausal inference')
    _persist_load_state_dict(nc_diffusion, _nc_ema_sd, strict=True)
else:
    print('  No EMA state_dict -- loading live diffusion weights for noncausal')
    _persist_load_state_dict(nc_diffusion, _ck_nc.get('diffusion_state_dict'), strict=True)

nc_diffusion.eval()
for _p in nc_diffusion.parameters():
    _p.requires_grad_(False)

_nc_core = getattr(nc_diffusion, '_orig_mod', nc_diffusion)
_nc_core = getattr(_nc_core, 'module', _nc_core)

# Stage 1 head + optional encoder/RCN (legacy contaminated ckpts still carry these keys)
nc_regression_head = None
nc_encoder = None
nc_rcn_runner = None
nc_stage1_variant = 'noncausal'
_rh_sd = _ck_nc.get('regression_head_state_dict')

if _rh_sd is not None and _is_regression_mean_predictor_sd(_rh_sd):
    nc_regression_head = RegressionMeanPredictor(
        RegressionPredictorConfig(
            in_channels=int(CONFIG_NC.rcn.driver_dim),
            out_channels=1,
            lr_height=int(CONFIG_NC.graph.lr_shape[0]),
            lr_width=int(CONFIG_NC.graph.lr_shape[1]),
            hr_height=int(CONFIG_NC.diffusion.height),
            hr_width=int(CONFIG_NC.diffusion.width),
            block_out_channels=(64, 128, 192),
            layers_per_block=2,
            norm_num_groups=16,
        )
    ).to(DEVICE)
    _persist_load_state_dict(nc_regression_head, _rh_sd, strict=True)
    nc_stage1_variant = 'noncausal'
    print('  regression_head = RegressionMeanPredictor (CorrDiff vanilla path)')
elif _rh_sd is not None:
    # Legacy ckpt_noncausal trained with encoder+RCN+GraphToGridDecoder (pre-protocol fix)
    _nc_rh_cfg = CONFIG_NC.two_stage.regression_head
    nc_regression_head = GraphToGridDecoder(
        d_model=int(_nc_rh_cfg.d_model),
        hr_h=int(CONFIG_NC.diffusion.height), hr_w=int(CONFIG_NC.diffusion.width),
        intermediate_h=int(_nc_rh_cfg.intermediate_h),
        intermediate_w=int(_nc_rh_cfg.intermediate_w),
        n_heads=int(_nc_rh_cfg.n_heads),
        refine_channels=int(_nc_rh_cfg.refine_channels),
        output_channels=1,
    ).to(DEVICE)
    _persist_load_state_dict(nc_regression_head, _rh_sd, strict=True)
    nc_stage1_variant = 'causal'
    print('  regression_head = GraphToGridDecoder (legacy causal Stage-1 ckpt)')

    if _ck_nc.get('encoder_state_dict') and _ck_nc.get('rcn_cell_state_dict'):
        _nc_allowed = set(builder_v5.dynamic_node_types) | set(builder_v5.static_node_types)
        _nc_enc_cfgs = []
        for _mp in CONFIG_NC.encoder.metapaths:
            if _mp.src in _nc_allowed and _mp.target in _nc_allowed:
                _nc_enc_cfgs.append(IntelligibleVariableConfig(
                    name=_mp.name, meta_path=(_mp.src, _mp.relation, _mp.target),
                    pool=_mp.get('pool', 'mean'),
                ))
        if pipeline_v5.get_static_dataset() is not None:
            _nc_enc_cfgs.append(IntelligibleVariableConfig(
                name='static', meta_path=('SP_HR', 'causes', 'GP850'), pool='mean',
            ))
        nc_encoder = IntelligibleVariableEncoder(
            configs=_nc_enc_cfgs,
            hidden_dim=int(CONFIG_NC.encoder.hidden_dim),
            conditioning_dim=int(CONFIG_NC.encoder.conditioning_dim),
        ).to(DEVICE)
        _nc_probe = builder_v5.lr_grid_to_nodes(NC_BATCHES[0]['lr'][0])
        nc_rcn_cell = RCNCell(
            num_vars=len(_nc_enc_cfgs),
            hidden_dim=int(CONFIG_NC.rcn.hidden_dim),
            driver_dim=_nc_probe.shape[-1],
            reconstruction_dim=_nc_probe.shape[-1],
            dropout=float(CONFIG_NC.rcn.dropout),
        ).to(DEVICE)
        nc_rcn_runner = RCNSequenceRunner(
            nc_rcn_cell, detach_interval=CONFIG_NC.rcn.get('detach_interval')
        )
        _persist_load_state_dict(nc_encoder, _ck_nc.get('encoder_state_dict'), strict=False)
        _persist_load_state_dict(nc_rcn_cell, _ck_nc.get('rcn_cell_state_dict'), strict=False)
        print('  legacy encoder+RCN loaded for GraphToGridDecoder inference')
    else:
        print('  WARNING: GraphToGridDecoder ckpt without encoder/RCN -- mu_HR may be wrong')
else:
    print('  No regression_head in noncausal_v4 checkpoint')

if nc_regression_head is not None:
    nc_regression_head.eval()
    for _p in nc_regression_head.parameters():
        _p.requires_grad_(False)
if nc_encoder is not None:
    nc_encoder.eval()
if nc_rcn_runner is not None:
    nc_rcn_runner.cell.eval()

print(f'[Cell 10] noncausal_v4 build complete  causal_concat={nc_causal_concat}  '
      f'stage1_variant={nc_stage1_variant}')


In [ ]:
# === Cell 11 : noncausal_v4 SAMPLE (idempotent) ===
from st_cdgm.evaluation.two_stage_inference import build_two_stage_inputs
from st_cdgm.evaluation.evaluation_xai import convert_sample_to_batch as _csb_nc

def _nc_build_inputs(batch):
    return build_two_stage_inputs(
        batch,
        variant=nc_stage1_variant,
        regression_head=nc_regression_head,
        encoder=nc_encoder,
        rcn_runner=nc_rcn_runner,
        builder=builder_v5,
        device=DEVICE,
    )

def _nc_sample_once(mu_HR, baseline_log):
    kw = dict(
        num_steps=NUM_STEPS,
        scheduler_type='edm_karras',
        cfg_scale=NONCAUSAL_CFG_SCALE,
        apply_constraints=False,
    )
    if nc_causal_concat and mu_HR is not None:
        kw['mu_HR'] = mu_HR
        kw['baseline_log'] = baseline_log
    return _nc_core.sample(conditioning=None, **kw).residual

if NONCAUSAL_PRED_CACHE.exists() and not SMOKE_MODE:
    print(f'[Cell 11] Loading cached noncausal_v4 predictions from {NONCAUSAL_PRED_CACHE}')
    _nc_cache = torch.load(str(NONCAUSAL_PRED_CACHE), map_location='cpu', weights_only=False)
    nc_pred_mm      = _nc_cache['pred_mm']
    nc_target_mm    = _nc_cache['target_mm']
    nc_pred_log     = _nc_cache['pred_full_log']
    nc_target_log   = _nc_cache['target_full_log']
    nc_pred_std     = _nc_cache['pred_std']
    nc_per_batch_rmse = _nc_cache['per_batch_rmse']
    print(f'[Cell 11] Loaded {nc_pred_mm.shape[0]} noncausal_v4 batches from cache')
else:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    _nc_means, _nc_stds, _nc_tgts, _nc_mus, _nc_bls, _nc_per_rmse = [], [], [], [], [], []
    _t0 = time.time()

    with torch.no_grad():
        for _bi, _sample in enumerate(NC_BATCHES):
            _batch = _csb_nc(_sample, builder_v5, DEVICE)
            _cond, _mu_HR, _bl, _tgt = _nc_build_inputs(_batch)

            _samples_k = []
            for _k in range(K_SAMPLES):
                torch.manual_seed(SEED + 3000 * _bi + _k)
                _res = _nc_sample_once(_mu_HR, _bl)
                _samples_k.append(_res)

            _stack = torch.stack(_samples_k, dim=0)
            _mean  = _stack.mean(dim=0)
            _std   = _stack.std(dim=0)

            _nc_means.append(_mean.cpu())
            _nc_stds.append(_std.cpu())
            _nc_tgts.append(_tgt.cpu())
            _nc_mus.append(_mu_HR.cpu() if _mu_HR is not None else torch.zeros_like(_mean.cpu()))
            _nc_bls.append(_bl.cpu())

            _nc_per_rmse.append(
                float(((_mean - _tgt).pow(2)[torch.isfinite(_tgt)].mean().sqrt()).item())
            )

            if (_bi + 1) % 4 == 0 or (_bi + 1) == len(NC_BATCHES):
                print(f'  [NC] batch {_bi+1}/{len(NC_BATCHES)} '
                      f'elapsed={time.time()-_t0:.0f}s')

    nc_pred_res = torch.cat(_nc_means, dim=0)
    nc_pred_std = torch.cat(_nc_stds, dim=0)
    nc_tgt_res  = torch.cat(_nc_tgts, dim=0)
    nc_mu_cat   = torch.cat(_nc_mus, dim=0)
    nc_bl_cat   = torch.cat(_nc_bls, dim=0)

    # Full reconstruction (noncausal_cell_061 BS31f pattern)
    if nc_causal_concat and torch.isfinite(nc_mu_cat).all():
        nc_pred_full = nc_mu_cat + nc_pred_res
    else:
        nc_pred_full = nc_pred_res

    nc_pred_log   = nc_bl_cat + nc_pred_full
    nc_target_log = nc_bl_cat + nc_tgt_res + nc_mu_cat

    nc_pred_mm   = to_mm_day(nc_pred_log)
    nc_target_mm = to_mm_day(nc_target_log)
    nc_per_batch_rmse = _nc_per_rmse

    if not SMOKE_MODE:
        torch.save({
            'pred_mm': nc_pred_mm, 'target_mm': nc_target_mm,
            'pred_full_log': nc_pred_log, 'target_full_log': nc_target_log,
            'pred_std': nc_pred_std, 'per_batch_rmse': nc_per_batch_rmse,
        }, str(NONCAUSAL_PRED_CACHE))

    print(f'[Cell 11] noncausal_v4 sampling done in {time.time()-_t0:.0f}s')
    print(f'[Cell 11] pred_mm shape = {tuple(nc_pred_mm.shape)}')


In [ ]:
# === Cell 12 : Convention B metrics (cGAN literature, pooled) ===
# Per noncausal_cell_061: pooled F1 + RMSE + MAE + Pearson + RAPSD + Spread + Drizzle.

print('=' * 80)
print('Convention B metrics (pooled, cGAN standard)')
print('=' * 80)

def _conv_B_for_model(model_name, pred_mm, target_mm, pred_std, per_batch_rmse, pred_log, target_log):
    results = {}
    pred_mm  = pred_mm.float()
    target_mm = target_mm.float()
    pred_log = pred_log.float()
    target_log = target_log.float()

    # F1 pooled (Convention B)
    _f1 = compute_f1_extremes(pred_mm, target_mm, threshold_percentiles=[95.0, 99.0])
    results['conv_B_F1p99'] = _f1.get('p99', float('nan'))
    results['conv_B_F1p95'] = _f1.get('p95', float('nan'))

    # RMSE, MAE, Spread (log1p residual space for direct comparison)
    _rmse, _mae, _spread = compute_rmse_mae_spread(pred_log, pred_std, target_log)
    results['conv_B_RMSE']   = _rmse
    results['conv_B_MAE']    = _mae
    results['conv_B_Spread'] = _spread

    # Pearson global + per-sample (noncausal_cell_061 lines 244-275)
    _cg, _cps, _ = compute_pearson_global_and_per_sample(pred_log, target_log)
    results['conv_B_Pearson_global']     = _cg
    results['conv_B_Pearson_per_sample'] = _cps

    # RAPSD distance (first 4 batches average, compute_spectrum_distance from eval_xai)
    results['conv_B_RAPSD'] = compute_rapsd_batch(pred_mm, target_mm, n_samples=4)

    # Drizzle bias (M5 fix: isfinite filter)
    results['conv_B_Drizzle_bias'] = drizzle_bias(
        pred_mm.numpy(), target_mm.numpy()
    )

    print(f'  [{model_name}]')
    for k, v in results.items():
        print(f'    {k:<32} = {v:.4f}' if isinstance(v, float) and v == v else f'    {k:<32} = {v}')
    return results

metrics_B = {}
metrics_B['phase8']       = _conv_B_for_model('phase8',       p8_pred_mm, p8_target_mm,
                                                p8_pred_std,  p8_per_batch_rmse,
                                                p8_pred_log, p8_target_log)
metrics_B['v5_causal']    = _conv_B_for_model('v5_causal',    v5_pred_mm, v5_target_mm,
                                                v5_pred_std,  v5_per_batch_rmse,
                                                v5_pred_log, v5_target_log)
metrics_B['noncausal_v4'] = _conv_B_for_model('noncausal_v4', nc_pred_mm, nc_target_mm,
                                                nc_pred_std,  nc_per_batch_rmse,
                                                nc_pred_log, nc_target_log)

print('\n[Cell 12] Convention B metrics computed')


In [ ]:
# === Cell 13 : Convention A metrics (ETCCDI WMO standard, per-pixel clim thresholds) ===

print('=' * 80)
print('Convention A metrics (ETCCDI per-pixel, WMO Zhang 2011)')
print('=' * 80)

def _conv_A_for_model(model_name, pred_mm, target_mm, pred_log=None, target_log=None):
    results = {}
    pred_np   = pred_mm.float().numpy()
    target_np = target_mm.float().numpy()
    N = pred_np.shape[0]

    # Reshape to (N, H_HR, W_HR) for pixel-wise ops
    try:
        pred_np   = pred_np.reshape(N, H_HR, W_HR)
        target_np = target_np.reshape(N, H_HR, W_HR)
    except Exception:
        pred_np   = pred_np.reshape(N, clim_p99_np.shape[0], clim_p99_np.shape[1])
        target_np = target_np.reshape(N, clim_p99_np.shape[0], clim_p99_np.shape[1])

    # F1 ETCCDI per-pixel p99 and p95 (Phase 8 Cell 13 exact pattern)
    _e99 = f1_etccdi_per_pixel(pred_np, target_np, land_mask_np, clim_p99_np)
    _e95 = f1_etccdi_per_pixel(pred_np, target_np, land_mask_np, clim_p95_np)
    results['conv_A_F1p99_etccdi'] = _e99['f1']
    results['conv_A_F1p95_etccdi'] = _e95['f1']
    results['conv_A_F1p99_tp']     = _e99['tp']
    results['conv_A_F1p99_fp']     = _e99['fp']
    results['conv_A_F1p99_fn']     = _e99['fn']

    # Binarize at clim_p99 per-pixel for CSI / SEDI / FSS
    valid_pix = land_mask_np & np.isfinite(clim_p99_np)
    pred_bin   = (pred_np >= clim_p99_np[None]) & valid_pix[None]
    target_bin = (target_np >= clim_p99_np[None]) & valid_pix[None]

    _csi, _sedi = csi_sedi(pred_bin, target_bin)
    results['conv_A_CSI_p99']  = _csi
    results['conv_A_SEDI_p99'] = _sedi

    # FSS at three neighbourhood scales (Phase 8 Cell 13)
    for _n in [9, 25, 51]:
        results[f'conv_A_FSS_p99_n{_n}'] = fss_neighborhood(pred_bin, target_bin, n=_n)

    # Climate indices bias (H6 label N-aware)
    _clim_idx = compute_climate_indices_bias(pred_np, target_np, land_mask_np, N)
    results.update(_clim_idx)

    print(f'  [{model_name}]')
    for k, v in results.items():
        if isinstance(v, float):
            print(f'    {k:<38} = {v:.4f}' if v == v else f'    {k:<38} = NaN')
        else:
            print(f'    {k:<38} = {v}')
    return results

metrics_A = {}
metrics_A['phase8']       = _conv_A_for_model('phase8',       p8_pred_mm, p8_target_mm,
                                                p8_pred_log, p8_target_log)
metrics_A['v5_causal']    = _conv_A_for_model('v5_causal',    v5_pred_mm, v5_target_mm,
                                                v5_pred_log, v5_target_log)
metrics_A['noncausal_v4'] = _conv_A_for_model('noncausal_v4', nc_pred_mm, nc_target_mm,
                                                nc_pred_log, nc_target_log)

print('\n[Cell 13] Convention A metrics computed')


In [ ]:
# === Cell 14 : Paired statistical tests (permutation + BCa bootstrap + Holm-Bonferroni) ===
import numpy as np
from path_c_plus.scripts.stats_utils import holm_bonferroni_correction

def _paired_permutation_test(a: np.ndarray, b: np.ndarray,
                              n_perm: int = PAIRED_PERMUTATION_N,
                              rng_seed: int = SEED) -> float:
    """Two-sided paired permutation test. Returns p-value."""
    rng = np.random.default_rng(rng_seed)
    diffs = a - b
    obs_stat = float(np.abs(np.mean(diffs)))
    count_ge = 0
    for _ in range(n_perm):
        signs = rng.choice([-1.0, 1.0], size=len(diffs))
        perm_stat = float(np.abs(np.mean(signs * diffs)))
        if perm_stat >= obs_stat:
            count_ge += 1
    return (count_ge + 1) / (n_perm + 1)


def _bca_bootstrap_ci(a: np.ndarray, b: np.ndarray,
                       n_boot: int = BOOTSTRAP_N_RESAMPLES,
                       confidence: float = 0.95,
                       rng_seed: int = SEED) -> dict:
    """BCa bootstrap CI for mean difference (a - b) per batch."""
    from path_c_plus.scripts.stats_utils import _norm_inv, _norm_cdf
    rng = np.random.default_rng(rng_seed)
    diffs = a - b
    n = len(diffs)
    point = float(np.mean(diffs))
    if n < 2:
        return {'point': point, 'ci_lo': float('nan'), 'ci_hi': float('nan')}
    boots = [float(np.mean(rng.choice(diffs, size=n, replace=True))) for _ in range(n_boot)]
    boots = np.array(boots)
    alpha = 1 - confidence
    # BCa
    z0 = float(np.clip(_norm_inv(np.mean(boots < point)), -3, 3))
    jack = np.array([np.mean(np.delete(diffs, i)) for i in range(n)])
    jm = jack.mean()
    num = np.sum((jm - jack) ** 3)
    den = 6 * (np.sum((jm - jack) ** 2)) ** 1.5
    a_acc = float(num / den) if den > 1e-12 else 0.0
    z_lo = _norm_inv(alpha / 2)
    z_hi = _norm_inv(1 - alpha / 2)
    a1 = np.clip(_norm_cdf(z0 + (z0 + z_lo) / (1 - a_acc * (z0 + z_lo))), 1e-3, 1 - 1e-3)
    a2 = np.clip(_norm_cdf(z0 + (z0 + z_hi) / (1 - a_acc * (z0 + z_hi))), 1e-3, 1 - 1e-3)
    ci_lo = float(np.percentile(boots, 100 * a1))
    ci_hi = float(np.percentile(boots, 100 * a2))
    return {'point': point, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'n_boot': n_boot}


def _per_batch_metric(pred_mm: torch.Tensor, target_mm: torch.Tensor) -> np.ndarray:
    """RMSE per batch sample for paired tests."""
    N = pred_mm.shape[0]
    out = []
    for i in range(N):
        v = torch.isfinite(target_mm[i]) & torch.isfinite(pred_mm[i])
        if v.sum() > 0:
            out.append(float(((pred_mm[i][v] - target_mm[i][v]) ** 2).mean().sqrt()))
        else:
            out.append(float('nan'))
    return np.array(out)


# Compute per-batch RMSE for each model
p8_batch_rmse = _per_batch_metric(p8_pred_mm, p8_target_mm)
v5_batch_rmse = _per_batch_metric(v5_pred_mm, v5_target_mm)
nc_batch_rmse = _per_batch_metric(nc_pred_mm, nc_target_mm)

print('[Cell 14] Running paired permutation tests...')
_pairs = [
    ('phase8_vs_v5_causal',    p8_batch_rmse, v5_batch_rmse),
    ('phase8_vs_noncausal_v4', p8_batch_rmse, nc_batch_rmse),
    ('v5_causal_vs_noncausal', v5_batch_rmse, nc_batch_rmse),
]

stat_tests = {}
_raw_pvals = []
_pair_keys = []

for _pair_name, _a, _b in _pairs:
    _valid = np.isfinite(_a) & np.isfinite(_b)
    _av = _a[_valid]; _bv = _b[_valid]
    if len(_av) < 2:
        print(f'  WARNING: {_pair_name} has <2 valid batch pairs, skipping')
        continue
    _pval = _paired_permutation_test(_av, _bv)
    _ci   = _bca_bootstrap_ci(_av, _bv)
    stat_tests[_pair_name] = {
        'metric': 'RMSE_per_batch',
        'n_valid_pairs': int(len(_av)),
        'mean_diff': float(_ci['point']),
        'ci_lo_95': float(_ci['ci_lo']),
        'ci_hi_95': float(_ci['ci_hi']),
        'p_permutation': float(_pval),
        'interpretation': 'negative diff = A better than B (lower RMSE)',
    }
    _raw_pvals.append(_pval)
    _pair_keys.append(_pair_name)
    print(f'  {_pair_name}: mean_diff={_ci["point"]:+.5f}  p={_pval:.4f}  '
          f'CI=[{_ci["ci_lo"]:.5f}, {_ci["ci_hi"]:.5f}]')

# Holm-Bonferroni correction
if _raw_pvals:
    _hb = holm_bonferroni_correction(_raw_pvals, alpha=0.05, labels=_pair_keys)
    for _k, _v in _hb.items():
        if _k in stat_tests:
            stat_tests[_k]['p_adjusted_holm'] = _v['p_adjusted']
            stat_tests[_k]['reject_H0_alpha05'] = _v['reject_at_alpha']
    print('\n  Holm-Bonferroni adjusted p-values:')
    for _k, _v in _hb.items():
        print(f'    {_k}: p_adj={_v["p_adjusted"]:.4f}  reject={_v["reject_at_alpha"]}')

print('\n[Cell 14] Statistical tests complete')


In [ ]:
# === Cell 15 : 3-way comparison table + final JSON ===
import json
from pathlib import Path

RESULTS_OUT = DRIVE_ROOT / 'oracle_9node' / 'eval_3way_dual_convention'
RESULTS_OUT.mkdir(parents=True, exist_ok=True)
FINAL_JSON  = RESULTS_OUT / 'results.json'

# ── Determine per-metric winner ──────────────────────────────────────────────
def _winner(m_p8, m_v5, m_nc, higher_is_better: bool = True) -> str:
    candidates = {'phase8': m_p8, 'v5_causal': m_v5, 'noncausal_v4': m_nc}
    valid = {k: v for k, v in candidates.items() if v == v}  # filter NaN
    if not valid:
        return 'unknown'
    if higher_is_better:
        return max(valid, key=lambda k: valid[k])
    else:
        return min(valid, key=lambda k: valid[k])

verdict = {}

# Convention B verdicts
for _metric, _hib in [
    ('conv_B_F1p99', True), ('conv_B_F1p95', True),
    ('conv_B_RMSE', False),  ('conv_B_MAE', False),
    ('conv_B_Pearson_global', True),
    ('conv_B_RAPSD', False),
]:
    _p8 = metrics_B['phase8'].get(_metric, float('nan'))
    _v5 = metrics_B['v5_causal'].get(_metric, float('nan'))
    _nc = metrics_B['noncausal_v4'].get(_metric, float('nan'))
    verdict[_metric] = {
        'phase8': _p8, 'v5_causal': _v5, 'noncausal_v4': _nc,
        'winner': _winner(_p8, _v5, _nc, _hib),
    }

# Convention A verdicts
for _metric, _hib in [
    ('conv_A_F1p99_etccdi', True), ('conv_A_F1p95_etccdi', True),
    ('conv_A_CSI_p99', True), ('conv_A_SEDI_p99', True),
    ('conv_A_FSS_p99_n9', True), ('conv_A_FSS_p99_n25', True), ('conv_A_FSS_p99_n51', True),
]:
    _p8 = metrics_A['phase8'].get(_metric, float('nan'))
    _v5 = metrics_A['v5_causal'].get(_metric, float('nan'))
    _nc = metrics_A['noncausal_v4'].get(_metric, float('nan'))
    verdict[_metric] = {
        'phase8': _p8, 'v5_causal': _v5, 'noncausal_v4': _nc,
        'winner': _winner(_p8, _v5, _nc, _hib),
    }

# Mark statistical significance in verdict
for _pair_key, _test in stat_tests.items():
    verdict.setdefault(f'stat_{_pair_key}', {}).update({
        'significant_at_alpha05': _test.get('reject_H0_alpha05', False),
        'p_adjusted': _test.get('p_adjusted_holm', float('nan')),
        'mean_diff': _test.get('mean_diff', float('nan')),
    })

# ── Print 3-way comparison table ────────────────────────────────────────────
print('\n' + '=' * 100)
print('3-WAY COMPARISON : Phase 8 vs V5_causal vs Noncausal_v4')
print('=' * 100)
print(f'  Test split: {K9_DATES["test"][0]} .. {K9_DATES["test"][1]}')
print(f'  N_batches={N_TEST_BATCHES}  K_samples={K_SAMPLES}  steps={NUM_STEPS}  seed={SEED}')
print()
_header = f'  {"Metric":<38} {"Phase8":>12} {"V5_causal":>12} {"Noncausal":>12}  Winner'
print(_header)
print('  ' + '-' * 82)

for _m in list(verdict.keys()):
    if _m.startswith('stat_'):
        continue
    _d = verdict[_m]
    _sig = ''
    for _pk in list(stat_tests.keys()):
        if stat_tests[_pk].get('reject_H0_alpha05', False):
            _sig = ' *'
            break
    _p8v = _d.get('phase8', float('nan'))
    _v5v = _d.get('v5_causal', float('nan'))
    _ncv = _d.get('noncausal_v4', float('nan'))
    _fmt = lambda v: f'{v:.4f}' if isinstance(v, float) and v == v else 'N/A'
    print(f'  {_m:<38} {_fmt(_p8v):>12} {_fmt(_v5v):>12} {_fmt(_ncv):>12}  {_d.get("winner","")}{_sig}')

# Count wins
_win_counts = {'phase8': 0, 'v5_causal': 0, 'noncausal_v4': 0, 'tied': 0}
for _m, _d in verdict.items():
    if _m.startswith('stat_'):
        continue
    _w = _d.get('winner', '')
    if _w in _win_counts:
        _win_counts[_w] += 1

print()
_conv_b_metrics = [m for m in verdict if m.startswith('conv_B') and not m.startswith('stat_')]
_conv_a_metrics = [m for m in verdict if m.startswith('conv_A') and not m.startswith('stat_')]
_p8_wins_b = sum(1 for m in _conv_b_metrics if verdict[m].get('winner') == 'phase8')
_p8_wins_a = sum(1 for m in _conv_a_metrics if verdict[m].get('winner') == 'phase8')
print(f'  Phase 8 wins: {_p8_wins_b}/{len(_conv_b_metrics)} metrics (Conv B)  '
      f'{_p8_wins_a}/{len(_conv_a_metrics)} metrics (Conv A)')
print(f'  (* = statistically significant at alpha=0.05 post Holm-Bonferroni)')

# ── Save final JSON ──────────────────────────────────────────────────────────
_results = {
    'pre_registration': PRE_REG,
    'metrics_B': metrics_B,
    'metrics_A': metrics_A,
    'stat_tests': stat_tests,
    'verdict_per_metric': verdict,
    'win_counts': _win_counts,
    'caveats': [
        'N_TEST_BATCHES is the eval_protocol.n_test_batches -- for full climate indices compute on all 730 test days.',
        'V5_causal and noncausal_v4 use a standard (non-9-node) graph builder; Phase 8 uses the 9-node extended builder.',
        'SHARED_BATCHES (Phase 8 test batches) and V5_BATCHES / NC_BATCHES are drawn from the same 2012-2013 period but via different LR pipeline files (19-var augmented vs 15-var canonical) -- temporal ordering may diverge by up to SEQ_LEN-1 steps.',
        'V5_causal architecture: ASSUMPTION that causal_concat=True and EMA key is diffusion_ema_state_dict or ema_state_dict. Verify if loading warnings appear.',
        'noncausal_v4 causal_concat is auto-detected from checkpoint; double-check printed value matches expectation.',
        'Drizzle_bias and climate indices are proxy estimates on N_TEST_BATCHES; full year required for ETCCDI compliance.',
    ],
}

FINAL_JSON.write_text(json.dumps(_results, indent=2, default=str))
print(f'\n[Cell 15] Final results JSON saved: {FINAL_JSON}')
print(f'[Cell 15] All done.')
